In [1]:

import glob
import pandas as pd
import os
import pyarrow as pa
import numpy as np
import pyarrow.parquet as pq
import re


from scipy.stats import spearmanr, pearsonr


In [2]:
from load_experiment_data import (
    train_dataset_name,
    test_dataset_name,
    train_dataset_split,
    test_dataset_split,
    load_data_and_estimators,
    explanation_types,
    explanation_m,
    linear_coders,
    explanation_k,
    explanation_lambda,
    explanation_seed
    )

/root/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/root/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `ty

In [3]:
NUM_TEST_INSTANCES = 1000 

In [4]:
k = len(explanation_k)
m = 1 # len(explanation_m)
lambda_ = len(explanation_lambda)
explanation_seed_ = 5

In [5]:
num_explanation_types = 1 # Self

num_explanation_types += k*m # explanations.TopKMostInfluential,
num_explanation_types += k*m # explanations.TopKLeastInfluential,
num_explanation_types += k*m # explanations.TopKMostHelpful,
num_explanation_types += k*m # explanations.TopKMostHarmful,
num_explanation_types += k*m*lambda_ # explanations.FacilityLocationMostHarmful,
num_explanation_types += k*m*lambda_ # explanations.FacilityLocationMostHelpful,
num_explanation_types += k*m*lambda_ # explanations.FacilityLocationMostInfluential,
num_explanation_types += k*m*lambda_ # explanations.FacilityLocationLeastInfluential,
num_explanation_types += k*m # explanations.DIVINEMostHelpful,
num_explanation_types += k*m # explanations.DIVINEMostHarmful,
num_explanation_types += k*m # explanations.DIVINEMostInfluential,
num_explanation_types += k*m # explanations.DIVINELeastInfluential,
num_explanation_types += k*m # explanations.AIDE
num_explanation_types += k*m*explanation_seed_
num_explanation_types

137

In [6]:
num_explanation_types_validation = num_explanation_types
num_explanation_types_validation -= 1 # AIDE

In [7]:
df_validation = pq.ParquetDataset("results/validation").read().to_pandas()


In [8]:
df_scoring = pq.ParquetDataset("results/scoring").read().to_pandas()

In [9]:
set(df_scoring["explanation_type"].unique()) - set(df_validation["explanation_type"].unique())

{'1 by facility location from Top-100 least influential (scores closest to zero). lambda=0.0',
 '1 by facility location from Top-100 least influential (scores closest to zero). lambda=0.5',
 '1 by facility location from Top-100 least influential (scores closest to zero). lambda=1.0',
 '1 by facility location from Top-100 most helpful (most negative scores). lambda=1.0',
 '25 by facility location from Top-100 least influential (scores closest to zero). lambda=0.5',
 '25 by facility location from Top-100 least influential (scores closest to zero). lambda=1.0',
 '25 by facility location from Top-100 most helpful (most negative scores). lambda=0.25',
 '25 by facility location from Top-100 most helpful (most negative scores). lambda=0.5'}

In [10]:
len(df_validation["explanation_type"].unique()) == num_explanation_types_validation

False

In [11]:
group_sizes = df_validation.groupby(
    ["train_dataset","train_split","test_dataset","test_split",
     "model","estimator","explanation_type"]
).size()

group_sizes[group_sizes != NUM_TEST_INSTANCES]

train_dataset                                 train_split  test_dataset                                  test_split  model                                                          estimator                                   explanation_type                                                                                       
loris3/tulu-3-sft-olmo-2-mixture-0225-sample  train        loris3/tulu-3-sft-olmo-2-mixture-0225-sample  test        OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42  BM25Estimator: k1=1.5, b=0.75               10 by facility location from Top-100 least influential (scores closest to zero). lambda=0.5                462
                                                                                                                                                                                    DataInfEstimator: fast_implementation=True  1 by facility location from Top-100 most helpful (most negative scores). lambda=0.25                       919
  

In [12]:
len(df_validation["explanation_type"].unique()) 

129

In [13]:
len(df_scoring["explanation_type"].unique()) == num_explanation_types

True

In [14]:
group_sizes = df_scoring.groupby(
    ["train_dataset","train_split","test_dataset","test_split",
     "model","estimator","linear_coder","explanation_type"]
).size()

group_sizes[group_sizes != NUM_TEST_INSTANCES]

Series([], dtype: int64)